In [ ]:
import numpy as np
import pandas as pd
import heapq
import random

In [ ]:
data=pd.read_csv('TripLegsTue.csv')
df=pd. read_excel('Fri16/MetroStatioد.xlsx')
data.dropna(inplace=True)

# assign line number:

In [ ]:
data = data.merge(df[['Station_ID','Line_ID']], how='left', left_on='Origin_Station_ID', right_on='Station_ID')
data = data.merge(df[['Station_ID','Line_ID']], how='left', left_on='Destination_Station_ID', right_on='Station_ID')
data.dropna(inplace=True)

In [ ]:
data=data.drop(['Station_ID_x','Station_ID_y'] , axis=1)
merged=data.rename({"Line_ID_x":"Line_ID_origin", "Line_ID_y": "Line_ID_dist"}, axis=1)

# determine the headway of each line:

In [ ]:
merged['GroupKey'] = merged['Time_Boarding'] // 60
grouped_data = merged.groupby('GroupKey').filter(lambda x: x['GroupKey'].nunique() == 1)
result = grouped_data.groupby('GroupKey').apply(lambda x: x.values.tolist()).to_dict()

In [ ]:
hour4, hour5,hour6, hour7,hour8, hour9, hour10,hour11, hour12, hour13,hour14, hour15, hour16, hour17,hour18,hour19, hour20,hour21, hour22, hour23 = [result[v] for v in result.keys()]
hours=[hour4, hour5,hour6, hour7,hour8, hour9, hour10,hour11, hour12, hour13,hour14, hour15, hour16, hour17,hour18,hour19, hour20,hour21, hour22, hour23]

In [ ]:
for i in range(len(hours)):
  hours[i]=pd.DataFrame(hours[i])
  hours[i]=hours[i].rename({ 0:"ETCARD_ID", 1: "Origin_Station_ID" ,2:"Time_Boarding" ,3:"Destination_Station_ID" ,4:"Time_Alighting" ,5:"Line_ID_origin" ,6:"Line_ID_dist" ,7:"hour_boarding" }, axis=1)

In [ ]:
headway={}
for j in range(len(hours)):
  groups_merged = hours[j].groupby('Destination_Station_ID')
  n=[]
  m=[]
  for name, group in groups_merged:
    diffs = group['Time_Alighting'].unique()
    diffs.sort()
    min_diff = diffs[1] - diffs[0] if len(diffs) > 1 else 0
    print(f"Minimum difference for group {j} with value {name}: {min_diff}")
    n.append(name)
    m.append(min_diff)
  headway[j]= [n,m]

In [ ]:
 for i in range(len(headway.keys())):
  f1=headway.get(i)
  # print(f1)
  transposed_data = list(zip(*f1))
  headway_data = pd.DataFrame(transposed_data, columns=['headway','station_Id'])

  # headway_data=pd.DataFrame(f1, columns)
  # print(headway_data)
  df_headway = df.merge(headway_data[['headway','station_Id']], how='left', left_on='Station_ID', right_on='station_Id')
  result = df_headway[df_headway['headway'] > 3].groupby('Line_ID')['headway'].min()
  print(f"Current key: {i}")
  print(result)
  print('-----------------')

# transfer:

In [ ]:
merged['transfer']=0

split line 1

In [ ]:
dataset1 = merged[(merged['Line_ID_origin'] == 1) | (merged['Line_ID_origin'] == 107) | (merged['Line_ID_origin'] == 104) | (merged['Line_ID_origin'] == 103) | (merged['Line_ID_origin'] == 102)]

In [ ]:
transfer_1 = dataset1[~dataset1['Line_ID_dist'].isin([1, 107, 103, 104, 102])]

In [ ]:
condition=merged['Line_ID_origin'].isin([1, 107, 103, 104, 102]) & ~merged['Line_ID_dist'].isin([1, 107, 103, 104, 102])

In [ ]:
merged.loc[condition, 'transfer'] = 1
#complete other lines like line 1 to determine that if each passenger has transfer or not and add that to our dataset(merged)

# passenger with transfer:

create network with all stations, determine shortest path and find transfer station:

In [ ]:
transfer = pd.concat([transfer_1, transfer_2, transfer_3, transfer_4], ignore_index=True) 
transfer = transfer.drop_duplicates()  
transfer.reset_index(drop=True, inplace=True)  

In [ ]:
n_station=pd.read_excel('Fri16/MetroStatioد.xlsx')
st=pd.read_excel('station.xlsx')

In [ ]:
transfer = pd.merge(transfer, n_station[['Station_ID', 'new_station_ID']], left_on='Origin_Station_ID', right_on='Station_ID', how='left') 
transfer = pd.merge(transfer, n_station[['Station_ID', 'new_station_ID']], left_on='Destination_Station_ID', right_on='Station_ID', how='left') 
transfer=transfer.drop(['Station_ID_x','Station_ID_y','Origin_Station_ID','Destination_Station_ID'] , axis=1).rename({"new_station_ID_x":"Origin_Station_ID", "new_station_ID_y": "Destination_Station_ID"}, axis=1)

In [ ]:
class Station:
    def __init__(self, name, lines):
        self.name = name
        self.lines = lines if isinstance(lines, list) else [lines]
        self.connections = {}

    def add_connection(self, station, distance):
        self.connections[station] = distance

class TrainNetwork:
    def __init__(self):
        self.stations = {}
        self.lines = {}

    def add_station(self, name, lines):
        if name not in self.stations:
            self.stations[name] = Station(name, lines)
            for line in (lines if isinstance(lines, list) else [lines]):
                if line not in self.lines:
                    self.lines[line] = []
                self.lines[line].append(name)

    def add_connection(self, station1, station2, distance):
        self.stations[station1].add_connection(station2, distance)
        self.stations[station2].add_connection(station1, distance)

    def add_stations_from_dataframe(self, df):
      for _, row in df.iterrows():
        station_id = row['station_ID']
        line_id = row['line_id_list']
        if isinstance(line_id, list):
            line_id_list = line_id
        else:
            line_id_list = [line_id]
        # filtered_list = list(filter(lambda x: x != 0, line_id_list))
        self.add_station(station_id, line_id_list)

    def dijkstra(self, start, end):
        distances = {station: float('inf') for station in self.stations}
        distances[start] = 0
        pq = [(0, start)]
        previous = {station: None for station in self.stations}

        while pq:
            current_distance, current_station = heapq.heappop(pq)

            if current_station == end:
                path = []
                while current_station:
                    path.append(current_station)
                    current_station = previous[current_station]
                return path[::-1], current_distance

            if current_distance > distances[current_station]:
                continue

            for neighbor, weight in self.stations[current_station].connections.items():
                distance = current_distance + weight
                if distance < distances[neighbor]:
                    distances[neighbor] = distance
                    previous[neighbor] = current_station
                    heapq.heappush(pq, (distance, neighbor))

        return None, float('inf')

In [ ]:
st['line_id_list'] = None   
for i in range(len(st)):  
    line_id = str(st['line_ID'][i]) 
    if len(line_id) == 3:  
        st.at[i, 'line_id_list'] = [line_id[0], line_id[2]]  
    else:  
        st.at[i, 'line_id_list'] = [st['line_ID'][i]]

In [ ]:
network = TrainNetwork()
network.add_stations_from_dataframe(st)

In [ ]:
for i in range(len(st)):
    k = i - 1 if i > 0 else 0 
    a = min(i + 2, len(st)) 
    for j in range(k,a):
        if abs(int(str(st['station_ID'][i])[len(str(st['line_ID'][i])):])-int(str(st['station_ID'][j])[len(str(st['line_ID'][j])):]))==1:
           network.add_connection(st['station_ID'][i], st['station_ID'][j], 2)

In [ ]:
network.add_connection(10216,215, 2)
network.add_connection(215,10216, 2)
network.add_connection(10216,213, 2)
network.add_connection(213,10216, 2)
network.add_connection(10612,617, 2)
network.add_connection(10612,615, 2)
network.add_connection(617,10612, 2)
network.add_connection(615,10612, 2)
network.add_connection(10414,49, 2)
network.add_connection(10414,20411, 2)
network.add_connection(49,10414, 2)
network.add_connection(20411,10414, 2)
network.add_connection(10310,313, 2)
network.add_connection(10310,311, 2)
network.add_connection(311,10310, 2)
network.add_connection(313,10310, 2)
network.add_connection(10719,30718, 2)
network.add_connection(30718,10719, 2)
network.add_connection(77,10719, 2)
network.add_connection(10719,77, 2)
network.add_connection(20610,614, 2)
network.add_connection(614,20610, 2)
network.add_connection(20610,4066, 2)
network.add_connection(4066,20610, 2)
network.add_connection(20411,4066, 2)
network.add_connection(4066,20411, 2)
network.add_connection(20718,40711, 2)
network.add_connection(40711,20718, 2)
network.add_connection(20718,713, 2)
network.add_connection(713,20718, 2)
network.add_connection(40515,20522, 2)
network.add_connection(20522,40515, 2)
network.add_connection(30615,619, 2)
network.add_connection(619,30615, 2)
network.add_connection(30615,617, 2)
network.add_connection(617,30615, 2)
network.add_connection(30416,410, 2)
network.add_connection(410,30416, 2)
network.add_connection(30416,49, 2)
network.add_connection(49,30416, 2)
network.add_connection(710,30718, 2)
network.add_connection(30718,710, 2)
network.add_connection(4066,611, 2)
network.add_connection(611,4066, 2)
network.add_connection(40711,716, 2)
network.add_connection(716,40711, 2)
network.add_connection(40515,53, 2)
network.add_connection(53,40515, 2)
network.add_connection(40617,626, 2)
network.add_connection(626,40617, 2)
network.add_connection(40617,628, 2)
network.add_connection(628,40617, 2)
network.add_connection(60710,76, 2)
network.add_connection(76,60710, 2)
network.add_connection(60710,74, 2)
network.add_connection(74,60710, 2)
network.add_connection(60721,716, 2)
network.add_connection(716,60721, 2)
network.add_connection(60721,718, 2)
network.add_connection(718,60721, 2)

In [ ]:
for i in range(len(transfer)):
    start=transfer['Origin_Station_ID'][i]
    end=transfer['Destination_Station_ID'][i]
    print(network.dijkstra(start, end))

In [ ]:
def find_transfer_stations(routes):
    def clean_station_id(station_id):
        station_str = str(station_id)
        if len(station_str) == 5 or  len(station_str) == 4:
            return station_str[:3].replace('0', '')
        return station_str[:1]
    
    transfer_stations = []
    
    for route in routes:
        # Initialize transfer station as None for this route
        route_transfer = None
        
        # Convert to numpy array for better performance
        stations = np.array(route[:])  
        # Process pairs of adjacent stations
        for i in range(1,len(stations)):
            if len(str(stations[i]))==5 or len(str(stations[i]))==4:
                curr_station= stations[i]
                pre_station = stations[i-1]
                if i < len(stations) - 1:  # This ensures that i + 1 is a valid index 
                    next_station = stations[i + 1]  
                else:    
                    break
                pre_clean = clean_station_id(pre_station)
                next_clean = clean_station_id(next_station)
                
                if not set(pre_clean) & set(next_clean):
                    route_transfer = curr_station
                    break
        
        transfer_stations.append(route_transfer)
    
    return transfer_stations
transfer['transfer_station']= find_transfer_stations(formatted_lists)

In [ ]:
transfer2 = transfer2.drop(null_rows.index).reset_index(drop=True) 

# determine the number of trains & reach & frequency for each line

In [ ]:
merged1=pd.merge(merged, n_station[['Station_ID', 'new_station_ID']], left_on='Origin_Station_ID', right_on='Station_ID', how='left') 
merged1=pd.merge(merged1, n_station[['Station_ID', 'new_station_ID']], left_on='Destination_Station_ID', right_on='Station_ID', how='left') 
merged1=merged1.drop(['Station_ID_x','Station_ID_y','Origin_Station_ID','Destination_Station_ID'] , axis=1).rename({"new_station_ID_x":"Origin_Station_ID", "new_station_ID_y": "Destination_Station_ID"}, axis=1)
merged2 = merged1.merge(transfer2[['ETCARD_ID','Time_Boarding','Origin_Station_ID','Destination_Station_ID', 'transfer_station']], on=['ETCARD_ID','Time_Boarding','Origin_Station_ID','Destination_Station_ID'], how='left')

In [ ]:
merged2['travel_time']=merged2['Time_Alighting']-merged2['Time_Boarding']
merged2['frequncy_individual']=merged2['travel_time']*60/10

In [ ]:
# line 1
dataset1_1=merged2[(merged2['Line_ID_origin'].isin([1, 107, 103, 104, 102]))&(merged2['Line_ID_dist']==1)]
dataset1_2=merged2[(merged2['Line_ID_origin'].isin([1, 107, 103, 104, 102]))&(merged2['Line_ID_dist'].isin([107, 103, 104, 102]))]
dataset1= pd.concat([dataset1_1, dataset1_2])

In [ ]:
dataset1['Destination_Station_ID'].value_counts()

In [ ]:
# choose one of the station with high value counts for each line (in line 1 is station 115)
df1=dataset1[(dataset1['Time_Alighting']<900)&(dataset1['Time_Alighting']>=450) & (dataset1['Destination_Station_ID']==115)]
df1=df1[(df1['Origin_Station_ID']<115)|(df1['Origin_Station_ID']==10612 )|(df1['Origin_Station_ID']==10414)|(df1['Origin_Station_ID']==10310)]

In [ ]:
df1['id']= range(len(df1))
y1=df1
y=y1['Time_Alighting']
x=y1['id']

In [ ]:
plt.figure(figsize=(10, 7))
plt.hist(y, bins=450, color='#21918c', edgecolor='black')
# sns.histplot(y, bins=90, kde=True, color='skyblue', edgecolor='black')
# Adding labels and title
xx= get_display(reshape('زمان'))
plt.xlabel(xx)
# plt.ylabel('Frequency')
label_fontsize=12
tt= get_display(reshape('خط 1'))
plt.title(tt,fontsize=label_fontsize)
plt.show()

In [ ]:
counts, bin_edges = np.histogram(y, bins=450)  
# print(counts)
# Find peaks in the histogram  
peaks, _ = find_peaks(counts)  

# Count the number of peaks  
num_peaks = len(peaks)  

# Print the number of peaks  
print(f'Number of peaks in the histogram: {num_peaks}')

In [ ]:
hist, bins = np.histogram(y, bins=145) 
bin_centers = 0.5 * (bins[1:] + bins[:-1])
distances = np.diff(bin_centers[peaks])  

# Compute mean distance  
mean_distance = np.mean(distances)  

# Output the mean distance  
print("Mean distance between peaks:", mean_distance)

In [ ]:
freq_line1=sum(dataset1['frequncy_individual'])/36
reach_line1=len(dataset1)/36
a1=dataset1['Origin_Station_ID'].value_counts()
# it can be done for other lines

# frequency for station in peak time

In [ ]:
# line 1
station_line1=merged2[merged2['Line_ID_origin'].isin([1, 107, 103, 104, 102])]
frequency_line1=3.91*60/10

# GRP for station in total

In [ ]:
# line 1
station_counts = data1['Origin_Station_ID'].value_counts()  

# If you want to save the station names and their counts separately  
stations = station_counts.index.tolist()  # List of station names  
counts = station_counts.values.tolist()
st1 = pd.DataFrame({'station': stations, 'count': counts})

In [ ]:
st1['GRP']=0
for i in range(len(st1)):
    st1['GRP'][i]=st1['count'][i]/len(merged2)*100*mean_distance*60/2/10

# frequency transfer station

In [ ]:
# line 1
transfer_station=merged2[~merged2['Line_ID_origin'].isin([1,2,3,4])]
station_counts=transfer_station[transfer_station['Line_ID_origin'].isin([107,103])]['Origin_Station_ID'].value_counts()

In [ ]:
stations = station_counts.index.tolist()  # List of station names  
counts = station_counts.values.tolist()
st_t1 = pd.DataFrame({'station': stations, 'count': counts})

In [ ]:
st_t1['GRP']=0
for i in range(len(st_t1)):
    st_t1['GRP'][i]=st_t1['count'][i]/len(merged2)*100*4.7*60/2/10

# complete dataset (based on survey)

In [ ]:
# augmentation
def generate_bits_with_percentages(num_bits, zero_percentage):  
    num_zeros = int(num_bits * zero_percentage)  
    num_ones = num_bits - num_zeros  
    bits = [0] * num_zeros + [1] * num_ones  
    random.shuffle(bits)  
    return bits  

In [ ]:
num_bits = 1101202 
women_percentage = 0.43
male= generate_bits_with_percentages(num_bits, women_percentage)  
sit_percentage = 0.27
siting= generate_bits_with_percentages(num_bits, sit_percentage)  

In [ ]:
data['male']=male
data['siting']=siting

In [ ]:
data.to_csv('final_dataset.csv')